In [1]:
# Step 1: Load shared training data

import pandas as pd
import numpy as np

from pathlib import Path

# Load the raw training features
X_train = pd.read_csv("../data/processed/X_train_raw.csv")

# Load training labels
y_train = pd.read_csv(
    "../data/processed/y_train.csv"
).squeeze("columns")

print("Training features:", X_train.shape)
print("Training labels:", y_train.shape)

print("\nChurn distribution:")
print(y_train.value_counts())

print("\nMissing values:", X_train.isnull().sum().sum())

print("\nFeature columns:")
print(X_train.columns.tolist())

Training features: (5634, 19)
Training labels: (5634,)

Churn distribution:
Churn
0    4139
1    1495
Name: count, dtype: int64

Missing values: 0

Feature columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


In [2]:
# Step 2: Build Logistic Regression Pipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Continuous numerical features
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

# Binary numerical feature
binary_features = ["SeniorCitizen"]

# Categorical features
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

# Define preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(
            handle_unknown="ignore"
        ), categorical_features),
        ("binary", "passthrough", binary_features)
    ]
)

# Build baseline Logistic Regression pipeline
lr_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

print("Logistic Regression Pipeline Created!")
print(lr_pipeline)

Logistic Regression Pipeline Created!
Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['gender', 'Partner',
                                                   'Dependents', 'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupp

In [3]:
# Step 3: Train Baseline Logistic Regression

# Train the complete pipeline
lr_pipeline.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

# Check how many numerical features the model uses
feature_names = lr_pipeline.named_steps[
    "preprocessing"
].get_feature_names_out()

print("\nNumber of model features:", len(feature_names))

# Generate predictions on training data
train_predictions = lr_pipeline.predict(X_train)

# Generate churn probabilities
train_probabilities = lr_pipeline.predict_proba(X_train)[:, 1]

print("\nFirst 5 predicted classes:")
print(train_predictions[:5])

print("\nFirst 5 churn probabilities:")
print(train_probabilities[:5])

Logistic Regression model trained successfully!

Number of model features: 45

First 5 predicted classes:
[0 0 0 0 0]

First 5 churn probabilities:
[0.34039619 0.28944793 0.04460955 0.09298389 0.37937896]


In [4]:
# Step 4: Stratified 5-Fold Cross-Validation

from sklearn.model_selection import StratifiedKFold, cross_validate

# Define stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Define evaluation metrics
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

# Run cross-validation
cv_results = cross_validate(
    lr_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Display results
print("=== BASELINE LOGISTIC REGRESSION ===")

for metric in scoring:
    scores = cv_results[f"test_{metric}"]

    print(
        f"{metric}: "
        f"{scores.mean():.4f} "
        f"(+/- {scores.std():.4f})"
    )

=== BASELINE LOGISTIC REGRESSION ===
accuracy: 0.8025 (+/- 0.0120)
precision: 0.6538 (+/- 0.0270)
recall: 0.5438 (+/- 0.0409)
f1: 0.5931 (+/- 0.0298)
roc_auc: 0.8462 (+/- 0.0126)
pr_auc: 0.6615 (+/- 0.0194)


In [5]:
# Step 5: Compare Standard and Class-Weighted Logistic Regression

from sklearn.base import clone
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import pandas as pd

# Standard baseline
standard_lr = clone(lr_pipeline)

# Class-weighted version
weighted_lr = clone(lr_pipeline)

weighted_lr.set_params(
    model__class_weight="balanced"
)

models = {
    "Standard Logistic Regression": standard_lr,
    "Weighted Logistic Regression": weighted_lr
}

comparison_results = []

for name, model in models.items():

    results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1
    )

    comparison_results.append({
        "Model": name,
        "Accuracy": results["test_accuracy"].mean(),
        "Precision": results["test_precision"].mean(),
        "Recall": results["test_recall"].mean(),
        "F1": results["test_f1"].mean(),
        "ROC-AUC": results["test_roc_auc"].mean(),
        "Average Precision": results["test_pr_auc"].mean()
    })

comparison_df = pd.DataFrame(comparison_results)

print("=== LOGISTIC REGRESSION COMPARISON ===")

print(comparison_df.round(4).to_string(index=False))

=== LOGISTIC REGRESSION COMPARISON ===
                       Model  Accuracy  Precision  Recall     F1  ROC-AUC  Average Precision
Standard Logistic Regression    0.8025     0.6538  0.5438 0.5931   0.8462             0.6615
Weighted Logistic Regression    0.7483     0.5167  0.8013 0.6281   0.8460             0.6599


In [6]:
# Step 6: Classification Threshold Analysis

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

# Generate out-of-fold probabilities using standard LR
cv_probabilities = cross_val_predict(
    standard_lr,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=1
)[:, 1]

# Test different thresholds
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

threshold_results = []

for threshold in thresholds:
    predictions = (cv_probabilities >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_train, predictions, zero_division=0
        ),
        "Recall": recall_score(y_train, predictions),
        "F1": f1_score(y_train, predictions)
    })

threshold_df = pd.DataFrame(threshold_results)

print("=== THRESHOLD COMPARISON ===")
print(threshold_df.round(4).to_string(index=False))

=== THRESHOLD COMPARISON ===
 Threshold  Precision  Recall     F1
       0.2     0.4751  0.8542 0.6106
       0.3     0.5413  0.7666 0.6346
       0.4     0.5935  0.6642 0.6269
       0.5     0.6535  0.5438 0.5936
       0.6     0.7102  0.3967 0.5090
       0.7     0.7822  0.2114 0.3328


In [7]:
# Step 7: Compare Logistic Regression configurations

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Standard LR probabilities already calculated in Step 6
standard_probs = cv_probabilities

# Weighted LR out-of-fold probabilities
weighted_probs = cross_val_predict(
    weighted_lr,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=1
)[:, 1]

configurations = {
    "Standard LR (0.50)": (standard_probs, 0.50),
    "Standard LR (0.30)": (standard_probs, 0.30),
    "Weighted LR (0.50)": (weighted_probs, 0.50)
}

results = []

for name, (probs, threshold) in configurations.items():

    predictions = (probs >= threshold).astype(int)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_train, predictions),
        "Precision": precision_score(
            y_train, predictions, zero_division=0
        ),
        "Recall": recall_score(y_train, predictions),
        "F1": f1_score(y_train, predictions),
        "ROC-AUC": roc_auc_score(y_train, probs),
        "Average Precision": average_precision_score(
            y_train, probs
        )
    })

final_comparison_df = pd.DataFrame(results)

print("=== LOGISTIC REGRESSION CONFIGURATION COMPARISON ===")
print(final_comparison_df.round(4).to_string(index=False))

=== LOGISTIC REGRESSION CONFIGURATION COMPARISON ===
             Model  Accuracy  Precision  Recall     F1  ROC-AUC  Average Precision
Standard LR (0.50)    0.8024     0.6535  0.5438 0.5936   0.8456             0.6585
Standard LR (0.30)    0.7657     0.5413  0.7666 0.6346   0.8456             0.6585
Weighted LR (0.50)    0.7483     0.5166  0.8013 0.6282   0.8455             0.6569


In [8]:
# Step 8: Logistic Regression Feature Coefficients

import pandas as pd
import numpy as np

# Use the standard LR pipeline trained in Step 3
trained_preprocessor = lr_pipeline.named_steps["preprocessing"]
trained_model = lr_pipeline.named_steps["model"]

# Get feature names after preprocessing
feature_names = trained_preprocessor.get_feature_names_out()

# Get Logistic Regression coefficients
coefficients = trained_model.coef_[0]

# Create a table
coefficient_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Odds_Ratio": np.exp(coefficients)
})

# Sort by coefficient
coefficient_df = coefficient_df.sort_values(
    "Coefficient",
    ascending=False
)

print("=== TOP 10 POSITIVE COEFFICIENTS ===")
print(coefficient_df.head(10).round(4).to_string(index=False))

print("\n=== TOP 10 NEGATIVE COEFFICIENTS ===")
print(coefficient_df.tail(10).round(4).to_string(index=False))

=== TOP 10 POSITIVE COEFFICIENTS ===
                            Feature  Coefficient  Odds_Ratio
   cat__InternetService_Fiber optic       0.6511      1.9176
       cat__Contract_Month-to-month       0.5761      1.7791
                  num__TotalCharges       0.5254      1.6911
               cat__StreamingTV_Yes       0.2084      1.2317
           cat__StreamingMovies_Yes       0.2083      1.2316
cat__PaymentMethod_Electronic check       0.1978      1.2187
             cat__OnlineSecurity_No       0.1568      1.1698
              binary__SeniorCitizen       0.1445      1.1555
                cat__TechSupport_No       0.1315      1.1405
             cat__MultipleLines_Yes       0.1099      1.1162

=== TOP 10 NEGATIVE COEFFICIENTS ===
                                  Feature  Coefficient  Odds_Ratio
 cat__StreamingMovies_No internet service      -0.3061      0.7363
cat__DeviceProtection_No internet service      -0.3061      0.7363
    cat__OnlineBackup_No internet service      -0.306

In [9]:
# Step 9: Save Logistic Regression model and results

from pathlib import Path
import joblib
import pandas as pd

model_dir = Path("../models")
reports_dir = Path("../reports")

model_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# Fit the standard model on all available training data
final_lr = clone(standard_lr)
final_lr.fit(X_train, y_train)

# Save the entire fitted pipeline
joblib.dump(
    final_lr,
    model_dir / "logistic_regression.joblib"
)

# Save exploratory cross-validation comparisons
final_comparison_df.to_csv(
    reports_dir / "logistic_regression_results.csv",
    index=False
)

# Save coefficients
coefficient_df.to_csv(
    reports_dir / "logistic_regression_coefficients.csv",
    index=False
)

# Save threshold separately
threshold_config = {
    "model": "Standard Logistic Regression",
    "threshold": 0.30,
    "status": "provisional",
    "selection_basis": "Highest exploratory F1 among tested thresholds"
}

joblib.dump(
    threshold_config,
    model_dir / "logistic_regression_config.joblib"
)

print("Logistic Regression model saved successfully!")
print("Comparison results saved successfully!")
print("Coefficients saved successfully!")
print("Provisional threshold configuration saved!")

Logistic Regression model saved successfully!
Comparison results saved successfully!
Coefficients saved successfully!
Provisional threshold configuration saved!


## Logistic Regression — Model Summary and Decision Log

### Objective
We developed Logistic Regression as the baseline classification model for predicting customer churn. The model estimates churn probabilities using customer information such as tenure, contract type, internet service, and monthly charges.

### Training approach
We used the shared stratified 80/20 train/test split, with 5,634 training customers and 1,409 customers reserved for final testing.

A scikit-learn Pipeline combined StandardScaler, OneHotEncoder, and Logistic Regression. Preprocessing was fitted separately within each cross-validation training fold to reduce data leakage.

### Model comparison

| Configuration | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| Standard LR (0.50) | 80.24% | 65.35% | 54.38% | 59.36% |
| Standard LR (0.30) | 76.57% | 54.13% | 76.66% | 63.46% |
| Weighted LR (0.50) | 74.83% | 51.66% | 80.13% | 62.82% |

### Model selection reasoning
The standard Logistic Regression model with a threshold of 0.30 achieved the highest observed F1-score among the tested configurations while maintaining relatively high churn recall.

It is retained as a provisional candidate for comparison with other algorithms. Because the threshold was selected using the same out-of-fold predictions used to report its performance, its F1-score may be optimistic.

### Feature interpretation
The fitted model showed positive coefficients for fiber optic internet, month-to-month contracts, and electronic check payments. Negative coefficients were observed for longer tenure and two-year contracts.

These coefficients describe associations within the model and do not establish the causes of churn. Correlation between variables also limits how independently some coefficients can be interpreted.

### Limitations
- The model has not yet been evaluated on the held-out test dataset.
- The selected threshold has not undergone independent validation.
- The model's probability calibration has not yet been assessed.
- The business cost of false positives and false negatives has not been quantified.

### Next steps
Compare Logistic Regression against Random Forest, Decision Tree, and XGBoost/LightGBM using consistent training-validation procedures. Select the final model and evaluate it on the untouched test dataset.

The selected model will then support high-risk customer profiling and feature explanations.